# This notebook uses a ResNet50 model

## Imports

In [53]:
import os
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, Subset, random_split

from torchvision import transforms, models

from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
from tqdm import tqdm

from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import numpy as np
from sklearn.model_selection import StratifiedKFold

from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

import pandas as pd


## Dataset class definition, inherited from PyTorch dataset class
A megadott Python osztály, ```MaskedImageDataset``` a PyTorch beépített Dataset osztályából származik, és képes képadatok és hozzájuk tartozó maszkok kezelésére. Az osztály ``__getitem__`` felülírt függvényének  kimenete az adathalmazban levő bináris maszkok felhasználásával adja vissza a maszkolt képeket. 

In [54]:
class MaskedImageDataset(Dataset):
    def __init__(self, data_dir, categories, transform=None, sample_size=None):
        self.image_paths = []
        self.mask_paths = []
        self.labels = []
        self.class_names = []
        self.transform = transform
        self.label_map = {cat: i for i, cat in enumerate(categories)}
        self.inv_label_map = {i: cat for cat, i in self.label_map.items()}  # Szám → név átalakítás

        for category in categories:
            images_path = os.path.join(data_dir, category, "images")
            masks_path = os.path.join(data_dir, category, "masks")
            
            if not os.path.isdir(images_path) or not os.path.isdir(masks_path):
                continue
            
            files = os.listdir(images_path)
            if sample_size:  # If sample_size is given, limit the number of files
                files = files[:sample_size]  
                
            for file in tqdm(files, desc=f"Loading {category} images"):
                img_path = os.path.join(images_path, file)
                mask_path = os.path.join(masks_path, file)
                
                if os.path.exists(mask_path):
                    self.image_paths.append(img_path)
                    self.mask_paths.append(mask_path)
                    self.labels.append(self.label_map[category])
                    self.class_names.append(category)

    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        mask_path = self.mask_paths[idx]
        label = self.labels[idx]
    
        # Load images and masks
        image = Image.open(img_path).convert("RGB")
        mask = Image.open(mask_path).convert("L")  # grayscale
    
        # Resize both image and mask
        fixed_size = (256, 256)
        image = image.resize(fixed_size, Image.BILINEAR)
        mask = mask.resize(fixed_size, Image.NEAREST)
    
        # Apply transforms
        if self.transform:
            image = self.transform(image)  # apply full transform to image
        mask = transforms.ToTensor()(mask)  # only convert mask to tensor
    
        # Convert mask to binary (0 or 1)
        mask = (mask > 0.5).float()
    
        # Ensure mask has 3 channels like the image
        mask = mask.expand(3, -1, -1)  # shape [3, 256, 256]
    
        # Ensure shape match
        assert image.shape == mask.shape, f"Shape mismatch: {image.shape} vs {mask.shape}"
    
        # Apply mask to image
        masked_image = image * mask
    
        return masked_image, label


### A MaskedImageDataset osztály használata

In [55]:
transform = transforms.Compose([
    transforms.Resize((256, 256)),  # ResNet bemeneti mérete
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
])

dataset = MaskedImageDataset(data_dir='/kaggle/input/covid19-radiography-database/COVID-19_Radiography_Dataset',
                             categories=["COVID", "Normal", "Viral Pneumonia", "Lung_Opacity"],
                             transform=transform,
                             sample_size=100)

num_classes = len(dataset.label_map)
print(num_classes)

Loading Lung_Opacity images: 100%|██████████| 100/100 [00:00<00:00, 706.42it/s]

4


### Adatok felosztása tanító és tesztelő halmazokra

In [56]:
# Define split ratios
train_ratio = 0.80
test_ratio = 0.20

total_size = len(dataset)

train_size = int(train_ratio * total_size)
test_size = int(test_ratio * total_size)

train_dataset, test_dataset = random_split(dataset, [train_size, test_size])

def count_per_category(dataset):
    category_counts = {cat: 0 for cat in dataset.dataset.label_map.keys()}
    
    # Wrap the loop with tqdm to show progress
    for _, label in tqdm(dataset, desc="Counting categories"):
        category_name = dataset.dataset.inv_label_map[label]
        category_counts[category_name] += 1
    
    return category_counts

# Example usage
print("Training Set Distribution:", count_per_category(train_dataset))
print("Test Set Distribution:", count_per_category(test_dataset))

Counting categories: 100%|██████████| 320/320 [00:02<00:00, 123.91it/s]


Training Set Distribution: {'COVID': 79, 'Normal': 79, 'Viral Pneumonia': 79, 'Lung_Opacity': 83}


Counting categories: 100%|██████████| 80/80 [00:00<00:00, 121.76it/s]

Test Set Distribution: {'COVID': 21, 'Normal': 21, 'Viral Pneumonia': 21, 'Lung_Opacity': 17}


## Konvolucios háló definiálása

A ResNet-50 (Residual Network 50) egy mély konvolúciós neurális hálózat, amelyet 2015-ben mutattak be a Microsoft Research Asia kutatói. Ez az architektúra a residual block koncepció köré épül, amely lehetővé teszi rendkívül mély hálózatok hatékony tanítását és használatát.

In [57]:
# model = CustomCNN(num_classes)
model = models.resnet50(pretrained=True)
# for param in model.parameters():
#     param.requires_grad = False  # Fagyasztjuk az alapmodelt

# Utolsó teljesen kapcsolt réteg (fc) cseréje saját osztályszámra
model.fc = nn.Sequential(
    nn.Linear(model.fc.in_features, 512),  # Első rejtett réteg
    nn.LeakyReLU(0.01),
    nn.Dropout(0.3),
    nn.Linear(512, 256),  # Második rejtett réteg
    nn.ReLU(),
    nn.Dropout(0.3),
    nn.Linear(256, num_classes)  # Kimeneti réteg
)


/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


## K-fold tanítási algoritmus metrikák számításával

In [58]:

class CustomSubset(Subset):
    def __init__(self, dataset, indices):
        super().__init__(dataset, indices)
        self.dataset = dataset  # Tároljuk az eredeti datasetet
        self.labels = [dataset[i][1] for i in indices]  # Labels az eredeti datasetből indexelve
    
    def __getitem__(self, idx):
        image, _ = self.dataset[self.indices[idx]]  # Kivesszük az adatot az eredeti datasetből
        return image, self.labels[idx]  # Az előre eltárolt labelt adjuk vissza


def train_kfold(model,
                dataset,
                num_classes=4,
                num_epochs=10,
                batch_size=32,
                lr=0.001,
                k_folds=5,
                early_stop_threshold=3,
                lr_reduce_factor=0.1,
                lr_patience=2):

    kfold = StratifiedKFold(n_splits=k_folds, shuffle=True, random_state=42)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    criterion = nn.CrossEntropyLoss()

    all_metrics = []
    all_histories = []

    labels = np.array([dataset[i][1] for i in range(len(dataset))])

    for fold, (train_idx, test_idx) in enumerate(kfold.split(np.zeros(len(dataset)), labels)):
        print(f'Fold {fold+1}/{k_folds}')

        val_size = len(test_idx) // 2
        val_idx = test_idx[:val_size]
        test_idx = test_idx[val_size:]

        train_subset = CustomSubset(dataset, train_idx)
        val_subset = CustomSubset(dataset, val_idx)
        test_subset = CustomSubset(dataset, test_idx)

        num_workers = os.cpu_count()

        train_loader = DataLoader(train_subset, batch_size=batch_size, shuffle=True, num_workers=num_workers)
        val_loader = DataLoader(val_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers)
        test_loader = DataLoader(test_subset, batch_size=batch_size, shuffle=False, num_workers=num_workers)

        model_fold = copy.deepcopy(model).to(device)
        optimizer = optim.Adam(model_fold.parameters(), lr=lr)

        # Initialize ReduceLROnPlateau scheduler
        scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=lr_reduce_factor,
                                      patience=lr_patience, verbose=True)

        best_model_wts = None
        best_acc = 0.0
        early_stop_count = 0

        history_fold = {
            'train_loss': [],
            'train_acc': [],
            'val_loss': [],
            'val_acc': [],
            'val_precision': [],
            'val_recall': [],
            'val_f1': [],
            'learning_rate': []
        }

        for epoch in range(num_epochs):
            model_fold.train()
            running_loss, correct, total = 0.0, 0, 0

            # Training phase
            for inputs_batch, labels_batch in train_loader:
                inputs_batch, labels_batch = inputs_batch.to(device), labels_batch.to(device)
                optimizer.zero_grad()
                outputs_batch = model_fold(inputs_batch)
                loss = criterion(outputs_batch, labels_batch)
                loss.backward()
                optimizer.step()

                running_loss += loss.item() * inputs_batch.size(0)
                _, predicted_batch = torch.max(outputs_batch, 1)
                correct += (predicted_batch == labels_batch).sum().item()
                total += labels_batch.size(0)

            train_loss_epoch = running_loss / total
            train_acc_epoch = correct / total

            # Validation phase
            model_fold.eval()
            val_running_loss, val_correct, val_total = 0.0, 0, 0
            y_true_val, y_pred_val = [], []

            with torch.no_grad():
                for inputs_val, labels_val in val_loader:
                    inputs_val, labels_val = inputs_val.to(device), labels_val.to(device)
                    outputs_val = model_fold(inputs_val)
                    loss_val_batch = criterion(outputs_val, labels_val)

                    val_running_loss += loss_val_batch.item() * inputs_val.size(0)
                    _, predicted_val = torch.max(outputs_val, 1)

                    val_correct += (predicted_val == labels_val).sum().item()
                    val_total += labels_val.size(0)

                    y_true_val.extend(labels_val.cpu().numpy())
                    y_pred_val.extend(predicted_val.cpu().numpy())

            val_loss_epoch = val_running_loss / val_total
            val_acc_epoch = val_correct / val_total

            class_report_val = classification_report(y_true_val,
                                                     y_pred_val,
                                                     output_dict=True,
                                                     zero_division=0)

            precision_per_class_val = {f'class_{cls}': class_report_val[str(cls)]['precision'] for cls in range(num_classes)}
            recall_per_class_val    = {f'class_{cls}': class_report_val[str(cls)]['recall'] for cls in range(num_classes)}
            f1_per_class_val        = {f'class_{cls}': class_report_val[str(cls)]['f1-score'] for cls in range(num_classes)}

            current_lr = optimizer.param_groups[0]['lr']

            print(f'Epoch {epoch+1}, Train Loss: {train_loss_epoch:.4f}, Train Acc: {train_acc_epoch:.4f}, Val Loss: {val_loss_epoch:.4f}, Val Acc: {val_acc_epoch:.4f}, LR: {current_lr:.6f}')

            # Scheduler step based on validation loss
            scheduler.step(val_loss_epoch)

            # Save epoch history
            history_fold['train_loss'].append(train_loss_epoch)
            history_fold['train_acc'].append(train_acc_epoch)
            history_fold['val_loss'].append(val_loss_epoch)
            history_fold['val_acc'].append(val_acc_epoch)
            history_fold['val_precision'].append(precision_per_class_val)
            history_fold['val_recall'].append(recall_per_class_val)
            history_fold['val_f1'].append(f1_per_class_val)
            history_fold['learning_rate'].append(current_lr)

            # Early stopping logic
            if val_acc_epoch > best_acc:
                best_acc = val_acc_epoch
                best_model_wts = copy.deepcopy(model_fold.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= early_stop_threshold:
                print(f"Early stopping at epoch {epoch+1}")
                break

        # Test phase after training is complete
        model_fold.load_state_dict(best_model_wts)
        model_fold.eval()

        y_true_test, y_pred_test = [], []

        with torch.no_grad():
            for inputs_test, labels_test in test_loader:
                inputs_test, labels_test = inputs_test.to(device), labels_test.to(device)
                outputs_test = model_fold(inputs_test)
                _, predicted_test = torch.max(outputs_test, 1)

                y_true_test.extend(labels_test.cpu().numpy())
                y_pred_test.extend(predicted_test.cpu().numpy())

        class_report_test = classification_report(y_true_test,
                                                  y_pred_test,
                                                  output_dict=True,
                                                  zero_division=0)

        precision_per_class_test = {f'class_{cls}': class_report_test[str(cls)]['precision'] for cls in range(num_classes)}
        recall_per_class_test    = {f'class_{cls}': class_report_test[str(cls)]['recall'] for cls in range(num_classes)}
        f1_per_class_test        = {f'class_{cls}': class_report_test[str(cls)]['f1-score'] for cls in range(num_classes)}

        all_metrics.append({
            'fold': fold + 1,
            'test_precision_per_class': precision_per_class_test,
            'test_recall_per_class': recall_per_class_test,
            'test_f1_per_class': f1_per_class_test,
            'confusion_matrix': confusion_matrix(y_true_test,y_pred_test).tolist()
        })

        all_histories.append(history_fold)

    torch.save(best_model_wts,'best_model.pth')
    print('Best model saved!')

    return all_metrics, all_histories


### K-fold alkalmazása

In [ ]:
metrics_results, histories_results = train_kfold(model,
                train_dataset,
                num_classes=4,
                num_epochs=5,
                batch_size=32,
                lr=0.001,
                k_folds=5,
                early_stop_threshold=5,
                lr_reduce_factor=0.1,
                lr_patience=3)


Fold 1/5


/usr/local/lib/python3.10/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1, Train Loss: 1.2505, Train Acc: 0.4023, Val Loss: 2.7868, Val Acc: 0.2500, LR: 0.001000
Epoch 2, Train Loss: 1.0373, Train Acc: 0.5391, Val Loss: 10.6003, Val Acc: 0.3125, LR: 0.001000
Epoch 3, Train Loss: 1.0990, Train Acc: 0.5898, Val Loss: 16.5154, Val Acc: 0.5938, LR: 0.001000
Epoch 4, Train Loss: 0.9419, Train Acc: 0.6250, Val Loss: 3.2359, Val Acc: 0.4375, LR: 0.001000
Epoch 5, Train Loss: 0.6778, Train Acc: 0.7305, Val Loss: 2.6169, Val Acc: 0.6250, LR: 0.001000
Fold 2/5


/usr/local/lib/python3.10/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1, Train Loss: 1.3339, Train Acc: 0.3594, Val Loss: 1.2333, Val Acc: 0.4062, LR: 0.001000
Epoch 2, Train Loss: 1.0444, Train Acc: 0.5078, Val Loss: 19.2044, Val Acc: 0.4062, LR: 0.001000
Epoch 3, Train Loss: 0.8512, Train Acc: 0.5820, Val Loss: 17.6017, Val Acc: 0.2500, LR: 0.001000
Epoch 4, Train Loss: 0.8474, Train Acc: 0.6172, Val Loss: 3.7565, Val Acc: 0.4062, LR: 0.001000
Epoch 5, Train Loss: 0.7781, Train Acc: 0.5859, Val Loss: 1.7611, Val Acc: 0.5000, LR: 0.001000
Fold 3/5


...

Epoch 1, Train Loss: 1.2808, Train Acc: 0.3828, Val Loss: 12.1671, Val Acc: 0.1875, LR: 0.001000
Epoch 2, Train Loss: 1.0591, Train Acc: 0.5156, Val Loss: 24.0112, Val Acc: 0.5000, LR: 0.001000
Epoch 3, Train Loss: 0.8875, Train Acc: 0.6250, Val Loss: 35.8271, Val Acc: 0.3750, LR: 0.001000
Epoch 4, Train Loss: 0.7424, Train Acc: 0.6719, Val Loss: 2.7281, Val Acc: 0.5000, LR: 0.001000
Epoch 5, Train Loss: 0.7273, Train Acc: 0.7305, Val Loss: 3.9038, Val Acc: 0.4688, LR: 0.001000
Fold 4/5


/usr/local/lib/python3.10/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1, Train Loss: 1.1503, Train Acc: 0.4688, Val Loss: 0.6105, Val Acc: 0.7188, LR: 0.001000
Epoch 2, Train Loss: 0.8421, Train Acc: 0.6758, Val Loss: 7.7443, Val Acc: 0.4375, LR: 0.001000
Epoch 3, Train Loss: 0.6879, Train Acc: 0.7539, Val Loss: 26.8857, Val Acc: 0.3750, LR: 0.001000
Epoch 4, Train Loss: 0.8843, Train Acc: 0.6719, Val Loss: 6.2293, Val Acc: 0.4688, LR: 0.001000
Epoch 5, Train Loss: 0.7441, Train Acc: 0.6953, Val Loss: 2.6205, Val Acc: 0.4688, LR: 0.001000
Fold 5/5


/usr/local/lib/python3.10/dist-packages/torch/optim/lr_scheduler.py:62: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn(


Epoch 1, Train Loss: 1.2909, Train Acc: 0.3750, Val Loss: 21.6779, Val Acc: 0.5000, LR: 0.001000


## Adatok mentése Google-Drive-ra

### Mentés mint excel állomány

In [ ]:
# Mappa elérési útja
results_dir = '/kaggle/working/results'

# Ellenőrizd, hogy a mappa létezik-e, és hozd létre, ha nem
if not os.path.exists(results_dir):
    os.makedirs(results_dir)
    print(f"Directory '{results_dir}' created.")

# Funkció az eredmények Excel fájlba mentéséhez
def save_results_to_excel(metrics_results, histories_results, filename="results.xlsx"):
    # Teljes fájlútvonal a results mappában
    filepath = os.path.join(results_dir, filename)
    
    # Pandas Excel író létrehozása
    with pd.ExcelWriter(filepath, engine='openpyxl') as writer:
        # Mentés: metrikák eredményei
        metrics_data = []
        for fold_metrics in metrics_results:
            fold = fold_metrics['fold']
            for cls in fold_metrics['test_precision_per_class']:
                metrics_data.append({
                    'Fold': fold,
                    'Class': cls,
                    'Precision': fold_metrics['test_precision_per_class'][cls],
                    'Recall': fold_metrics['test_recall_per_class'][cls],
                    'F1-Score': fold_metrics['test_f1_per_class'][cls]
                })

        metrics_df = pd.DataFrame(metrics_data)
        metrics_df.to_excel(writer, sheet_name="Metrics", index=False)

        # Mentés: tanulási történetek (histories)
        history_data = []
        for fold_idx, history_fold in enumerate(histories_results):
            for epoch in range(len(history_fold['train_loss'])):
                history_data.append({
                    'Fold': fold_idx + 1,
                    'Epoch': epoch + 1,
                    'Train Loss': history_fold['train_loss'][epoch],
                    'Train Accuracy': history_fold['train_acc'][epoch],
                    'Validation Loss': history_fold['val_loss'][epoch],
                    'Validation Accuracy': history_fold['val_acc'][epoch],
                    'Learning Rate': history_fold['learning_rate'][epoch]
                })

        histories_df = pd.DataFrame(history_data)
        histories_df.to_excel(writer, sheet_name="Histories", index=False)

    print(f"Results saved to {filepath}")

# Example usage
save_results_to_excel(metrics_results, histories_results, "ResNet50_results_AllData.xlsx")


In [ ]:
import os
from google.oauth2 import service_account
from googleapiclient.discovery import build
from googleapiclient.http import MediaFileUpload

# JSON kulcs fájl elérési útja
SERVICE_ACCOUNT_FILE = '/kaggle/input/googledriveuploadauth/covidcxr-0aaa95b10e20.json'

# Hitelesítés
credentials = service_account.Credentials.from_service_account_file(
    SERVICE_ACCOUNT_FILE,
    scopes=['https://www.googleapis.com/auth/drive']
)

# Google Drive API kliens létrehozása
drive_service = build('drive', 'v3', credentials=credentials)

# Kaggle mappa elérési útja (ahonnan fájlokat töltünk fel)
source_dir = '/kaggle/working/results'

# Szülőmappa ID a Google Drive-on (ahol az új mappa létrejön)
parent_folder_id = '1LVJ2nsLuiiOB_4qJP5xbgRgYl-cs6MTU'  # Cseréld ki a megfelelő szülőmappa ID-ra

# Google Drive célmappa neve (paraméterként megadva)
target_folder_name = 'results'

# Ellenőrizd, hogy a forrásmappa létezik-e
if not os.path.exists(source_dir):
    print(f"Hiba: A '{source_dir}' mappa nem létezik.")
else:
    # Listázd az összes fájlt a mappában
    files = [os.path.join(source_dir, f) for f in os.listdir(source_dir) if os.path.isfile(os.path.join(source_dir, f))]

    # Ellenőrizd, hogy létezik-e a célmappa a megadott szülőmappában; ha nem, hozd létre
    def get_or_create_folder(folder_name, parent_id):
        # Keressük meg a mappát név és szülő ID alapján
        query = f"name='{folder_name}' and mimeType='application/vnd.google-apps.folder' and '{parent_id}' in parents"
        results = drive_service.files().list(q=query, fields="files(id, name)").execute()
        items = results.get('files', [])
        
        if items:
            # Ha létezik, térjünk vissza az ID-jával
            return items[0]['id']
        else:
            # Ha nem létezik, hozzuk létre
            file_metadata = {
                'name': folder_name,
                'mimeType': 'application/vnd.google-apps.folder',
                'parents': [parent_id]  # Szülőmappa ID-ja
            }
            folder = drive_service.files().create(body=file_metadata, fields='id').execute()
            return folder.get('id')

    # Célmappa ID lekérése vagy létrehozása a szülőmappában
    folder_id = get_or_create_folder(target_folder_name, parent_folder_id)

    # Fájlok feltöltése az újonnan létrehozott célmappába
    for file_path in files:
        file_name = os.path.basename(file_path)  # Fájl neve
        file_metadata = {
            'name': file_name,
            'parents': [folder_id]  # Célmappa azonosítója
        }
        media = MediaFileUpload(file_path, mimetype='application/octet-stream')  # MIME típus általános fájlokra

        try:
            file = drive_service.files().create(
                body=file_metadata,
                media_body=media,
                fields='id'
            ).execute()
            print(f"Feltöltve: {file_name} (File ID: {file.get('id')})")
        except Exception as e:
            print(f"Hiba történt a '{file_name}' feltöltése során: {e}")
